In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from students.degtyarev.lesson2 import Exercise

iris_data, iris_target = load_iris(return_X_y=True)
target_binary = (iris_target == 0).astype(int)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    iris_data, target_binary, test_size=0.2, random_state=42, stratify=target_binary
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

x_mean = X_train.mean(axis=0)
x_std = X_train.std(axis=0)
X_train_n = (X_train - x_mean) / x_std
X_val_n = (X_val - x_mean) / x_std
X_test_n = (X_test - x_mean) / x_std

learning_rates = [0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
batch_sizes = [4, 8, 16, 32, 64, None]
seeds = [1, 2, 3, 4, 5]
n_epochs = 50

results = []

print(f"{'LR':<7} | {'Batch':<6} | {'Mean F1':<8} | {'Stability':<9}")
print("-" * 40)

for lr in learning_rates:
    for bs in batch_sizes:
        f1_scores = []
        for s in seeds:
            model = Exercise.create_logistic_model(num_features=X_train.shape[1], rng=np.random.default_rng(s))
            Exercise.fit(model, X_train_n, y_train, lr=lr, n_iter=n_epochs, batch_size=bs)

            current_f1 = model.f1_score(X_val_n, y_val)
            f1_scores.append(current_f1)

        avg_f1 = np.mean(f1_scores)
        std_f1 = np.std(f1_scores)
        stability = avg_f1 - std_f1

        results.append({"lr": lr, "bs": bs, "f1": avg_f1, "stability": stability})

        bs_str = str(bs) if bs else "Full"
        print(f"{lr:<7} | {bs_str:<6} | {avg_f1:<8.4f} | {stability:<9.4f}")

best_config = sorted(results, key=lambda x: x["stability"], reverse=True)[0]

print("\n" + "=" * 40)
print("ФИНАЛЬНЫЙ ВЫБОР:")
print(f"Лучший Learning Rate: {best_config['lr']}")
print(f"Лучший Batch Size:   {best_config['bs']}")
print(f"Показатель стабильности: {best_config['stability']:.4f}")
print("=" * 40)

final_model = Exercise.create_logistic_model(num_features=X_train.shape[1])
Exercise.fit(final_model, X_train_n, y_train, lr=best_config["lr"], n_iter=100, batch_size=best_config["bs"])

print("\nМетрики на тестовых данных (X_test):")
print(f"Accuracy:  {final_model.accuracy(X_test_n, y_test):.4f}")
print(f"F1-score:  {final_model.f1_score(X_test_n, y_test):.4f}")
print(f"AUROC:     {final_model.auroc(X_test_n, y_test):.4f}")

LR      | Batch  | Mean F1  | Stability
----------------------------------------
0.001   | 4      | 0.8539   | 0.7616   
0.001   | 8      | 0.4617   | 0.3238   
0.001   | 16     | 0.2336   | 0.0713   
0.001   | 32     | 0.1781   | 0.0242   
0.001   | 64     | 0.1610   | 0.0229   
0.001   | Full   | 0.1289   | 0.0012   
0.01    | 4      | 1.0000   | 1.0000   
0.01    | 8      | 1.0000   | 1.0000   
0.01    | 16     | 1.0000   | 1.0000   
0.01    | 32     | 0.9379   | 0.8772   
0.01    | 64     | 0.7891   | 0.6898   
0.01    | Full   | 0.3904   | 0.2049   
0.05    | 4      | 1.0000   | 1.0000   
0.05    | 8      | 1.0000   | 1.0000   
0.05    | 16     | 1.0000   | 1.0000   
0.05    | 32     | 1.0000   | 1.0000   
0.05    | 64     | 1.0000   | 1.0000   
0.05    | Full   | 1.0000   | 1.0000   
0.1     | 4      | 1.0000   | 1.0000   
0.1     | 8      | 1.0000   | 1.0000   
0.1     | 16     | 1.0000   | 1.0000   
0.1     | 32     | 1.0000   | 1.0000   
0.1     | 64     | 1.0000   | 1.0000   